In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ModuleNotFoundError:
    print('Google Colab not detected; skipping Drive mount.')

In [ ]:
import pandas as pd
from pathlib import Path

# Primary base requested by user, with container fallbacks.
DATA_DIR_CANDIDATES = [
    Path('/workspace/Data'),
    Path('/workspaces/ns-final-proj/Data'),
    Path.cwd() / 'Data',
    Path.cwd().parent / 'Data',
]

MODELS_DIR_CANDIDATES = [
    Path('/workspace/Models'),
    Path('/workspaces/ns-final-proj/Models'),
    Path.cwd() / 'Models',
    Path.cwd().parent / 'Models',
]

def first_existing_dir(candidates):
    for path in candidates:
        if path.exists():
            return path
    return candidates[0]

DATA_DIR = first_existing_dir(DATA_DIR_CANDIDATES)
MODELS_DIR = first_existing_dir(MODELS_DIR_CANDIDATES)
MODELS_DIR.mkdir(parents=True, exist_ok=True)

def find_csv(filename: str):
    for root in DATA_DIR_CANDIDATES:
        candidate = root / filename
        if candidate.exists():
            return candidate
    return None

dataset_path = find_csv('OS_Scan_dataset.csv')
labels_path = find_csv('OS_Scan_labels.csv')

if dataset_path and labels_path:
    OS_Scan_data_total = pd.read_csv(dataset_path)
    OS_Scan_label_total = pd.read_csv(labels_path)
else:
    # Fallback so notebook can run even when OS_Scan files are not mounted.
    fallback_specs = [
        ('kitsune_selected_fn_rows.csv', 0),
        ('kitsune_selected_fp_rows.csv', 0),
        ('kitsune_selected_tp_rows.csv', 1),
    ]
    fallback_frames = []
    fallback_labels = []

    for name, label_value in fallback_specs:
        csv_path = find_csv(name)
        if csv_path is not None:
            df_part = pd.read_csv(csv_path)
            fallback_frames.append(df_part)
            fallback_labels.extend([label_value] * len(df_part))

    if not fallback_frames:
        checked_dirs = [str(p) for p in DATA_DIR_CANDIDATES]
        raise FileNotFoundError(
            "No input CSV files found. Expected OS_Scan_dataset.csv and OS_Scan_labels.csv, "
            "or fallback kitsune_selected_*.csv files. "
            f"Checked: {checked_dirs}"
        )

    OS_Scan_data_total = pd.concat(fallback_frames, ignore_index=True)
    OS_Scan_label_total = pd.DataFrame({
        'idx': range(len(fallback_labels)),
        'label': fallback_labels,
    })

OS_Scan_data_total

In [ ]:
# Labels are loaded in cell 2
OS_Scan_label_total

In [ ]:
import numpy as np

def pick_case(indices, label, preferred_index):
    if not indices:
        print(f'No {label} available yet. Returning NaN.')
        return None
    idx = min(preferred_index, len(indices) - 1)
    value = indices[idx]
    print(f'Using {label} sample at position {idx} of {len(indices)}')
    return value

def safe_array_value(values, idx, label='values'):
    if idx is None:
        return np.nan
    if idx < 0 or idx >= len(values):
        print(f'{label} index {idx} out of range for size {len(values)}. Returning NaN.')
        return np.nan
    return values[idx]

def safe_series_value(series, idx, label='series'):
    if idx is None:
        return np.nan
    if idx < 0 or idx >= len(series):
        print(f'{label} index {idx} out of range for size {len(series)}. Returning NaN.')
        return np.nan
    return series.iloc[idx]

def safe_window(length, center_idx, lookback=10):
    if center_idx is None:
        return 0, 0
    start_idx = max(0, center_idx - lookback)
    end_idx = min(length, center_idx)
    return start_idx, end_idx

In [ ]:
# Drop the last row
OS_Scan_label_total = OS_Scan_label_total.drop(OS_Scan_label_total.index[-1])
OS_Scan_label_total

In [ ]:
# Add labels as a new column to the train dataset
OS_Scan_data_total['labels'] = OS_Scan_label_total.iloc[:, 1]

# Save the updated train dataset to a new CSV file
merged_csv_path = DATA_DIR / 'merged_OS_Scan.csv'
# OS_Scan_data_total.to_csv(merged_csv_path, index=False)


In [ ]:
import psutil

# Get disk usage information
disk_usage = psutil.disk_usage('/')

# Print the used disk space in bytes
print("Used disk space:", disk_usage.used)

# Print the used disk space in a human-readable format
print("Used disk space:", psutil.disk_usage('/').used / (1024**3), "GB")


In [ ]:
# Calculate the distribution of each label
# Get the label column (assuming it is the last column)
label_column = OS_Scan_data_total.iloc[:, -1]
# Calculate the distribution of each label
label_distribution = label_column.value_counts(normalize=True)

# Print the label distribution
print(label_distribution)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from tensorflow import keras
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense
from sklearn.model_selection import train_test_split

# Assuming you have your data loaded into the data_array
# Remove the first column from the data
feature_data = OS_Scan_data_total.iloc[:, :-1]
label_data = OS_Scan_data_total.iloc[:, -1]
#del OS_Scan_data_total
#del OS_Scan_label_total

# Calculate the index for splitting the data
n_rows = len(feature_data)
if n_rows < 2:
    raise ValueError('Need at least 2 rows to split train/test data.')
split_index = min(1000000, max(1, int(n_rows * 0.8)))
if split_index >= n_rows:
    split_index = n_rows - 1

from sklearn.preprocessing import RobustScaler

# Create a RobustScaler object
scaler = RobustScaler()

# Normalize each column using RobustScaler
normalized_data = scaler.fit_transform(feature_data)

# Create a new DataFrame with the normalized values
df = pd.DataFrame(normalized_data, columns=feature_data.columns)

from sklearn.preprocessing import MinMaxScaler

# Assuming your DataFrame is called 'df'
# Create an instance of MinMaxScaler
scaler = MinMaxScaler()

# Fit the scaler on the entire DataFrame
scaler.fit(df)

# Transform the entire DataFrame with the scaler
normalized_df = pd.DataFrame(scaler.transform(df), columns=df.columns)

# Split the data into training and testing sets
X_train = normalized_df[:split_index]
X_test = normalized_df[split_index:]
Y_test = label_data[split_index:]


In [ ]:
X_train = X_train.to_numpy()
X_test = X_test.to_numpy()

In [ ]:

del normalized_data
del OS_Scan_data_total
del feature_data



In [ ]:
# Define the autoencoder architecture with additional layers
input_dim = X_train.shape[1]
encoding_dim = 100  # Adjust the size of the encoding layer as per your requirements

input_data = keras.Input(shape=(input_dim,))
encoded = keras.layers.Dense(256, activation='relu')(input_data)
dropout_encoded = keras.layers.Dropout(0.1)(encoded)
encoded2 = keras.layers.Dense(128, activation='relu')(dropout_encoded)
dropout_encoded2 = keras.layers.Dropout(0.1)(encoded2)
encoded3 = keras.layers.Dense(64, activation='relu')(dropout_encoded2)
dropout_encoded3 = keras.layers.Dropout(0.1)(encoded3)
encoded4 = keras.layers.Dense(128, activation='relu')(dropout_encoded3)
dropout_encoded4 = keras.layers.Dropout(0.1)(encoded4)
encoded5 = keras.layers.Dense(256, activation='relu')(dropout_encoded4)
dropout_encoded5 = keras.layers.Dropout(0.1)(encoded5)
decoded = keras.layers.Dense(input_dim, activation='sigmoid')(dropout_encoded5)

autoencoder = keras.Model(input_data, decoded)

from tensorflow.keras.optimizers import Adam

# Assuming 'autoencoder' is the model you want to compile
learning_rate = 0.001  # Learning rate value

# Create an optimizer with the desired learning rate
optimizer = Adam(learning_rate=learning_rate)
autoencoder.compile(optimizer=optimizer, loss='mse')

# Train the autoencoder
autoencoder.fit(X_train, X_train, epochs=10, batch_size=5000)

# Save the model in native Keras format (recommended in Keras 3).
model_path = MODELS_DIR / 'kitsune.keras'
autoencoder.save(model_path)
print(f"Model saved to {model_path}")

# Load the model without re-compiling legacy training config.
loaded_model = keras.models.load_model(model_path, compile=False)
print("Model loaded.")

# Reconstruction
reconstructed_data = autoencoder.predict(X_test)

# Compute reconstruction error
reconstruction_error = np.mean(np.square(X_test - reconstructed_data), axis=1)

# Set the threshold for anomaly detection
threshold = np.mean(reconstruction_error) + 2 * np.std(reconstruction_error)

# Classify anomalies
predicted_labels = (reconstruction_error > threshold).astype(int)

# Evaluate performance
accuracy = np.mean(predicted_labels == Y_test)

print(f"Accuracy: {accuracy}")


In [ ]:


# Reconstruction
reconstructed_data1 = autoencoder.predict(normalized_df)

# Compute reconstruction error
reconstruction_error1 = np.mean(np.square(normalized_df - reconstructed_data1), axis=1)

In [ ]:
# Create scatter plot
required_vars = ['reconstruction_error1', 'label_data', 'threshold', 'split_index']
missing = [name for name in required_vars if name not in globals()]
if missing:
    raise RuntimeError(f"Run prerequisite cells first. Missing variables: {missing}")

fig, ax = plt.subplots()

# Keep arrays aligned even when using fallback datasets.
plot_len = min(len(reconstruction_error1), len(label_data))
x_values = range(plot_len)
y_values = reconstruction_error1[:plot_len]
c_values = label_data.iloc[:plot_len]

ax.scatter(x_values, y_values, c=c_values, cmap='coolwarm')

# Set axis labels and title
ax.set_xlabel('Index')
#ax.set_yscale('log')
ax.set_ylabel('Reconstruction Error')
ax.set_title('Reconstruction Error vs. Index')

# Plot threshold and split markers
ax.axhline(y=threshold, color='red', linestyle='--')
ax.axvline(x=split_index, color='blue', linestyle='-.')

# Show the plot
plt.show()

In [ ]:
import psutil

# Get disk usage information
disk_usage = psutil.disk_usage('/')

# Print the used disk space in bytes
print("Used disk space:", disk_usage.used)

# Print the used disk space in a human-readable format
print("Used disk space:", psutil.disk_usage('/').used / (1024**3), "GB")

In [ ]:
# Prepare data for explanations
threshold

In [ ]:
predicted_labels

In [ ]:
Y_test = label_data[split_index:]

In [ ]:
kitsune_false_positives = []  # Store indices of false positives
kitsune_false_negatives = []  # Store indices of false negatives
kitsune_true_positives = []  # Store indices of false negatives
for i in range(len(reconstruction_error)):
    if predicted_labels[i] != Y_test.values[i]:
        if predicted_labels[i] == 1:  # False positive
            kitsune_false_positives.append(i)
        else:  # False negative
            kitsune_false_negatives.append(i)
    else:
        if predicted_labels[i] == 1:  # True positive
            kitsune_true_positives.append(i)

In [ ]:
kfn = pick_case(kitsune_false_negatives, 'false-negative', 39000)
kfn if kfn is not None else np.nan

In [ ]:
safe_array_value(reconstruction_error, kfn, 'reconstruction_error')

In [ ]:
safe_series_value(Y_test, kfn, 'Y_test')

In [ ]:
label_idx = (split_index + kfn) if kfn is not None else None
safe_series_value(label_data, label_idx, 'label_data')

In [ ]:
start_idx, end_idx = safe_window(len(X_test), kfn, lookback=10)
if end_idx <= start_idx:
    print('FN export skipped: no rows available for selected index.')
    kitsune_selected_fn_rows = pd.DataFrame()
else:
    kitsune_selected_fn_rows = pd.DataFrame(X_test[start_idx:end_idx])
    kitsune_selected_fn_rows.to_csv(DATA_DIR / 'kitsune_selected_fn_rows.csv', index=False)
kitsune_selected_fn_rows

In [ ]:
ktp = pick_case(kitsune_true_positives, 'true-positive', 20000)
ktp if ktp is not None else np.nan

In [ ]:
safe_array_value(reconstruction_error, ktp, 'reconstruction_error')

In [ ]:
safe_series_value(Y_test, ktp, 'Y_test')

In [ ]:
label_idx = (split_index + ktp) if ktp is not None else None
safe_series_value(label_data, label_idx, 'label_data')

In [ ]:
start_idx, end_idx = safe_window(len(X_test), ktp, lookback=10)
if end_idx <= start_idx:
    print('TP export skipped: no rows available for selected index.')
    kitsune_selected_tp_rows = pd.DataFrame()
else:
    kitsune_selected_tp_rows = pd.DataFrame(X_test[start_idx:end_idx])
    kitsune_selected_tp_rows.to_csv(DATA_DIR / 'kitsune_selected_tp_rows.csv', index=False)
kitsune_selected_tp_rows

In [ ]:
len(kitsune_false_positives)

In [ ]:
kfp = pick_case(kitsune_false_positives, 'false-positive', 20000)
kfp if kfp is not None else np.nan

In [ ]:
safe_array_value(reconstruction_error, kfp, 'reconstruction_error')

In [ ]:
start_idx, end_idx = safe_window(len(X_test), kfp, lookback=10)
if end_idx <= start_idx:
    print('FP export skipped: no rows available for selected index.')
    kitsune_selected_fp_rows = pd.DataFrame()
else:
    kitsune_selected_fp_rows = pd.DataFrame(X_test[start_idx:end_idx])
    # Save DataFrame as CSV
    kitsune_selected_fp_rows.to_csv(DATA_DIR / 'kitsune_selected_fp_rows.csv', index=False)
kitsune_selected_fp_rows